<div dir="rtl">

# 🗄️ 01 - InMemoryVectorStore in LangChain Core

## ما هو الـ Vector Store (مستودع المتجهات)؟
- **Vector Store** هو قاعدة بيانات متخصصة في تخزين وإدارة متجهات التضمين (**Embeddings**) للنصوص والمستندات مع البيانات الوصفية الخاصة بها (**Metadata**).
- يتيح البحث فائق السرعة عن المستندات الأكثر شبهاً دلالياً (**Semantic Similarity Search**) لأي استعلام بحثي.

---

### 💡 ما هو `InMemoryVectorStore`؟
- مستودع متجهات مدمج أصلياً في نواة LangChain (`langchain_core.vectorstores`).
- يحتفظ بالمتجهات في ذاكرة الوصول العشوائي (RAM) أثناء تشغيل الكود فقط.
- **أهم ميزاته**:
  - خفيف جداً ولا يتطلب تثبيت أي قواعد بيانات أو حزم خارجية معقدة.
  - مثالي للاختبارات السريعة (Prototyping & Unit Testing) والتطبيقات المؤقتة (Ephemeral Sessions).

</div>


<div dir="rtl">

### 1️⃣ تجهيز البيئة ونموذج التضمين (Embeddings)
نستخدم نموذج التضمين المحلي `sentence-transformers/all-MiniLM-L6-v2` عبر `HuggingFaceEmbeddings`.

</div>


In [ ]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore

# تحميل متغيرات البيئة
load_dotenv(find_dotenv())

# تهيئة نموذج التضمين
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("✅ تم تحميل نموذج التضمين بنجاح!")


<div dir="rtl">

### 2️⃣ إنشاء مستندات تجريبية (Sample Documents)
نقوم بإنشاء مجموعة من المستندات مع بيانات وصفية (`metadata`) لتوضيح الفئات والموضوعات.

</div>


In [ ]:
documents = [
    Document(
        page_content="الذكاء الاصطناعي التوليدي والشبكات العصبية العميقة تُحدث ثورة في معالجة اللغات الطبيعية.",
        metadata={"category": "AI", "topic": "GenAI", "year": 2024}
    ),
    Document(
        page_content="تقنيات RAG تساعد النماذج اللغوية الكبيرة على استرجاع معلومات دقيقة من قواعد بيانات خارجية.",
        metadata={"category": "AI", "topic": "RAG", "year": 2024}
    ),
    Document(
        page_content="تتميز لغة بايثون بسهولة تعلمها ووفرة مكتباتها المتخصصة في علوم البيانات والذكاء الاصطناعي.",
        metadata={"category": "Programming", "topic": "Python", "year": 2023}
    ),
    Document(
        page_content="تعتبر ممارسة الرياضة اليومية وتناول الغذاء الصحي من أهم أسباب الحفاظ على النشاط البدني.",
        metadata={"category": "Health", "topic": "Fitness", "year": 2022}
    ),
    Document(
        page_content="الحمية المتوسطية تعتمد على زيت الزيتون والخضار والأسماك وتساهم في صحة القلب.",
        metadata={"category": "Health", "topic": "Nutrition", "year": 2023}
    )
]

print(f"عدد المستندات المجهزة: {len(documents)}")


<div dir="rtl">

### 3️⃣ تهيئة `InMemoryVectorStore` وإضافة المستندات
يمكن إنشاء المستودع وتضمين النصوص مباشرة عبر `from_documents`.

</div>


In [ ]:
# إنشاء مستودع الذاكرة وتضمين المستندات
vector_store = InMemoryVectorStore.from_documents(
    documents=documents,
    embedding=embeddings
)

print("✅ تم إنشاء InMemoryVectorStore وتضمين كافة المستندات في الذاكرة بنجاح!")


<div dir="rtl">

### 4️⃣ البحث الدلالي البسيط (Similarity Search)
البحث عن أفضل المستندات المطابقة دلالياً للاستعلام.

</div>


In [ ]:
query = "كيف يمكن ربط النماذج اللغوية بمصادر بيانات للحصول على إجابات دقيقة؟"

results = vector_store.similarity_search(query, k=2)

print(f"🔍 الاستعلام: {query}\n")
for i, doc in enumerate(results, 1):
    print(f"--- النتيجة {i} ---")
    print(f"المحتوى: {doc.page_content}")
    print(f"الميتاداتا: {doc.metadata}\n")


<div dir="rtl">

### 5️⃣ البحث مع حساب درجات التشابه (Similarity Search with Score)
لحساب درجة الشبه أو المسافة لكل مستند بالنسبة للاستعلام.

</div>


In [ ]:
results_with_scores = vector_store.similarity_search_with_score(
    "ما هي الأطعمة الصحية المفيدة للجسم والقلب؟",
    k=3
)

for doc, score in results_with_scores:
    print(f"درجة التطابق / المسافة: {score:.4f}")
    print(f"المحتوى: {doc.page_content}")
    print(f"الميتاداتا: {doc.metadata}")
    print("-" * 50)


<div dir="rtl">

### 6️⃣ الفلترة باستخدام البيانات الوصفية (Metadata Filtering)
تصفية النتائج بناءً على شروط الميتاداتا (مثلاً: استرجاع النتائج من فئة Health فقط).

</div>


In [ ]:
# دالة تصفية مخصصة
def health_filter(doc: Document) -> bool:
    return doc.metadata.get("category") == "Health"

filtered_results = vector_store.similarity_search(
    "ما هي أفضل الممارسات للحياة اليومية والنشاط؟",
    k=2,
    filter=health_filter
)

print("🎯 نتائج البحث المفلترة (Health فقط):")
for doc in filtered_results:
    print(f"• {doc.page_content} (Category: {doc.metadata['category']})")
